# Exp7.4 — Latent Softmax Evidence Ablation

Aggregate-only analysis notebook. It reads the finalized Exp7.4 artifacts and does **not** train models or submit jobs.

Primary question: how much performance changes when the Exp7.3 A2 direct linear head is replaced by a factorized head, a latent softmax bottleneck, and a softmax bottleneck with deterministic RMS magnitude restoration.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
if not (repo / 'notebooks').exists():
    repo = repo.parent
base = repo / 'notebooks' / 'artifacts' / 'experiment_7_4_latent_softmax_evidence' / 'latent_softmax_evidence_v1'
manifest = json.loads((base / 'manifest.json').read_text())
runs = pd.read_csv(base / 'method_runs.csv')
summary = pd.read_csv(base / 'method_summary.csv')
contrast_runs = pd.read_csv(base / 'contrast_runs.csv')
contrasts = pd.read_csv(base / 'contrast_summary.csv')
manifest


## 1. Main result — test balanced accuracy

In [ ]:
main_cols = [
    'method', 'head', 'head_trainable_parameters',
    'test_ba_mean', 'test_ba_std',
    'l2_wholecount_probe_test_ba_mean',
    'l2_fixed250_probe_test_ba_mean',
    'best_epoch_mean',
]
main = summary[main_cols].copy()
for col in ['test_ba_mean', 'test_ba_std', 'l2_wholecount_probe_test_ba_mean', 'l2_fixed250_probe_test_ba_mean']:
    main[col] = 100 * main[col]
main.rename(columns={
    'test_ba_mean': 'test_BA_%',
    'test_ba_std': 'test_BA_std_pp',
    'l2_wholecount_probe_test_ba_mean': 'L2_wholecount_probe_BA_%',
    'l2_fixed250_probe_test_ba_mean': 'L2_fixed250_probe_BA_%',
}, inplace=True)
main.round(2)

In [ ]:
plot_df = summary[['method', 'test_ba_mean', 'test_ba_std']].copy()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(plot_df['method'], 100 * plot_df['test_ba_mean'], yerr=100 * plot_df['test_ba_std'], capsize=4)
ax.set_ylabel('Test balanced accuracy (%)')
ax.set_xlabel('Method')
ax.set_title('Exp7.4: main classification result')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
plt.show()

## 2. Paired mechanistic contrasts

These are the comparisons the experiment was designed to isolate:

- **B − A**: cost of replacing the direct 128→12 head with two linear matrices.
- **C − B**: effect of inserting the per-timestep 128-way softmax normalization.
- **D − C**: recovery after restoring deterministic latent RMS magnitude.
- **D − B**: residual gap between softmax+RMS and the matched two-linear control.

In [ ]:
contrast_view = contrasts[[
    'contrast', 'left_method', 'right_method',
    'test_ba_delta_pp_mean', 'test_ba_delta_pp_std',
    'l2_wholecount_probe_delta_pp_mean', 'l2_wholecount_probe_delta_pp_std',
]].copy()
contrast_view.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(contrasts['contrast'], contrasts['test_ba_delta_pp_mean'], yerr=contrasts['test_ba_delta_pp_std'], capsize=4)
ax.axhline(0, linewidth=1)
ax.set_ylabel('Paired test BA delta (percentage points)')
ax.set_title('Mechanistic contrasts')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
plt.show()

## 3. L2 representation quality

The two probe columns test whether end-to-end training changes how linearly accessible the class information remains at L2. This helps separate a head-only effect from representation damage.

In [ ]:
probe_cols = [
    'method',
    'l2_wholecount_probe_test_ba_mean', 'l2_wholecount_probe_test_ba_std',
    'l2_fixed250_probe_test_ba_mean', 'l2_fixed250_probe_test_ba_std',
]
probe = summary[probe_cols].copy()
for col in probe.columns:
    if col != 'method':
        probe[col] = 100 * probe[col]
probe.round(2)

## 4. Softmax and magnitude diagnostics

In [ ]:
diag_cols = [
    'method',
    'test_latent_rms_mean_mean',
    'test_softmax_entropy_mean_mean',
    'test_softmax_entropy_normalized_mean_mean',
    'test_softmax_max_prob_mean_mean',
    'test_softmax_effective_states_mean_mean',
    'test_magnitude_rms_mean_mean',
    'test_evidence_rms_mean_mean',
]
summary[diag_cols].round(3)

## 5. Current interpretation from the completed runs

Using the finalized three-seed aggregate currently in the artifact directory:

- **A — direct A2 head:** 56.45% test BA.
- **B — two linear matrices:** 51.11% test BA, a **−5.34 pp** drop vs A. So the factorized 128→128→12 parameterization is already not neutral in end-to-end training, even though it is algebraically linear.
- **C — latent softmax:** 25.74% test BA, another **−25.37 pp** vs B. This is the dominant degradation and strongly supports the hypothesis that per-timestep softmax normalization removes information the classifier was using.
- **D — softmax + RMS magnitude:** 39.26% test BA, recovering **+13.52 pp** vs C. Therefore magnitude is clearly important, but RMS magnitude alone does not recover the full information lost by the softmax bottleneck; D remains **−11.85 pp** below B.

The L2 probes also deteriorate from A to B/C/D, so part of the loss is not merely at the final head: the end-to-end objective changes the learned L2 representation itself.

The main mechanistic conclusion is therefore:

$$\boxed{\text{softmax loses substantially more than just a scalar evidence magnitude}}$$

Restoring one scalar magnitude recovers a large fraction of the loss, but not all of it. The remaining gap is consistent with information loss from simplex normalization itself (for example common-mode / logit-offset information and detailed signed relative amplitudes), plus end-to-end representation adaptation.

## 6. Per-seed sanity check

In [ ]:
runs[['method', 'seed', 'test_ba', 'l2_wholecount_probe_test_ba', 'l2_fixed250_probe_test_ba', 'best_epoch']].sort_values(['method', 'seed']).round(4)